# Desafío · Una fuente cambió sin avisar

La aplicación publicó una versión nueva. El lote contiene un tipo de evento nuevo (`refund`), una columna adicional (`app_version`) y al menos un evento duplicado. Incorporalo sin perder la historia ni duplicar identificadores.

In [0]:
dbutils.widgets.text("student_id", "alumno99", "Identificador del alumno")
dbutils.widgets.dropdown("scale", "small", ["test", "small", "demo"], "Escala")

In [0]:
%run ../common/config

In [0]:
%run ../common/generate_data

In [0]:
catalog = spark.sql("SELECT current_catalog()").first()[0]
config = CourseConfig(catalog, dbutils.widgets.get("student_id"), dbutils.widgets.get("scale"))
create_course_namespace(spark, config)
challenge_path = f"{config.volume_path}/events_json_v2"
build_challenge_batch(spark, config).coalesce(2).write.mode("overwrite").json(challenge_path)
print(challenge_path)

/Volumes/workspace/bigdata_alumno99/landing/events_json_v2


## Consigna 1 — Contrato y diagnóstico

Leé el lote nuevo, compará su esquema con `bronze_events` y respondé:

1. ¿Qué campo apareció?
2. ¿Qué valor de `event_type` no existía?
3. ¿Cuántas filas e identificadores distintos contiene?

In [0]:
# TODO: leer challenge_path, inspeccionar el esquema y calcular las tres respuestas.
challenge = None
eventsv2_raw = spark.read.json(challenge_path)
eventsv2_raw.printSchema()

root
 |-- app_version: string (nullable = true)
 |-- context: struct (nullable = true)
 |    |-- platform: string (nullable = true)
 |    |-- session_id: string (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- event_id: long (nullable = true)
 |-- event_ts: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- product_id: long (nullable = true)



## Consigna 2 — Integración controlada

Construí `bronze_events_v2` con estas propiedades:

- Contiene los eventos históricos y el lote nuevo.
- Conserva `app_version`, usando `NULL` para la historia anterior.
- Tiene un único registro por `event_id`.
- Agrega `_source`, `_source_file` y `_ingested_at`.
- Puede volver a ejecutarse sin cambiar el total final.

No modifiques `bronze_events`: queremos conservarlo como evidencia de la primera ingesta.

In [0]:
quality_summary = eventsv2_raw.agg(
    F.count("*").alias("rows"),
    F.countDistinct("event_id").alias("distinct_ids"),
)
display(quality_summary)

rows,distinct_ids
10001,10000


In [0]:
eventsv2_clean = eventsv2_raw.dropDuplicates(["event_id"])

In [0]:
def with_ingestion_metadata(frame, source_name, source_path):
    return (frame
        .withColumn("_source", F.lit(source_name))
        .withColumn("_source_file", F.lit(source_path))
        .withColumn("_ingested_at", F.current_timestamp()))

eventsv2_w_metadata = with_ingestion_metadata(eventsv2_clean, "events_json_v2", challenge_path)
eventsv2_w_metadata.printSchema()

root
 |-- app_version: string (nullable = true)
 |-- context: struct (nullable = true)
 |    |-- platform: string (nullable = true)
 |    |-- session_id: string (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- event_id: long (nullable = true)
 |-- event_ts: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- product_id: long (nullable = true)
 |-- _source: string (nullable = false)
 |-- _source_file: string (nullable = false)
 |-- _ingested_at: timestamp (nullable = false)



In [0]:
from pyspark.sql import functions as F

source_table_fp = f"{config.catalog}.{config.schema}.bronze_events"
target_table_fp = f"{config.catalog}.{config.schema}.bronze_events_v2"

# TODO: unir por nombre admitiendo columnas faltantes, deduplicar y escribir target_table.
source_table = spark.read.table(source_table_fp)

eventsv2_w_metadata.unionByName(source_table, allowMissingColumns=True).dropDuplicates(["event_id"]).write.mode("overwrite").option("mergeSchema", "true").saveAsTable(target_table_fp)


## Consigna 3 — Verificación automática

Completá las aserciones. Una entrega válida debe demostrar que no hay duplicados y que la columna evolucionada existe.

In [0]:
# TODO: reemplazar los valores pendientes.
result = spark.read.table(target_table_fp)
total_rows = result.count()
distinct_ids = result.select('event_id').distinct().count()
assert total_rows == distinct_ids, "Hay event_id duplicados"
assert "app_version" in result.columns, "No se preservó la evolución del esquema"
assert result.where("event_type = 'refund'").count() > 0, "No se incorporó refund"

## Reflexión final

En no más de 150 palabras, explicá la diferencia entre: detectar una evolución de esquema, aceptarla técnicamente y decidir que es válida para el negocio.